In [7]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import classification_report

In [8]:
# Define path
train_dir = "data/train"
test_dir = "data/test"
validation_dir = "data/val"
# define batch size, image size
img_height, img_width = 224, 224
batch_size = 24

train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size = (img_height, img_width),
    batch_size = batch_size
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size = (img_height, img_width),
    batch_size = batch_size
)

val_data = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size = (img_height, img_width),
    batch_size = batch_size
)

class_names = train_data.class_names
print(class_names)
print(len(class_names))

for i, j in train_data.take(1):
    print(i.shape,j.shape)

Found 6225 files belonging to 11 classes.
Found 3187 files belonging to 11 classes.
Found 1092 files belonging to 11 classes.
['animal fish', 'animal fish bass', 'fish sea_food black_sea_sprat', 'fish sea_food gilt_head_bream', 'fish sea_food hourse_mackerel', 'fish sea_food red_mullet', 'fish sea_food red_sea_bream', 'fish sea_food sea_bass', 'fish sea_food shrimp', 'fish sea_food striped_red_mullet', 'fish sea_food trout']
11
(24, 224, 224, 3) (24,)


In [9]:
AUTOTUNE = tf.data.AUTOTUNE

normalization_layer = layers.Rescaling(1./255)

train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
val_data   = val_data.map(lambda x, y: (normalization_layer(x), y))
test_data  = test_data.map(lambda x, y: (normalization_layer(x), y))

train_data = train_data.prefetch(AUTOTUNE)
val_data   = val_data.prefetch(AUTOTUNE)
test_data  = test_data.prefetch(AUTOTUNE)


In [10]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])


In [11]:
def build_tl_model(base_model, num_classes):
    base_model.trainable = False

    model = models.Sequential([
        data_augmentation,
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


In [12]:
num_classes = len(class_names)

models_dict = {
    "VGG16": tf.keras.applications.VGG16(
        weights="imagenet", include_top=False, input_shape=(224,224,3)
    ),
    "ResNet50": tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_shape=(224,224,3)
    ),
    "MobileNet": tf.keras.applications.MobileNet(
        weights="imagenet", include_top=False, input_shape=(224,224,3)
    ),
    "InceptionV3": tf.keras.applications.InceptionV3(
        weights="imagenet", include_top=False, input_shape=(224,224,3)
    ),
    "EfficientNetB0": tf.keras.applications.EfficientNetB0(
        weights="imagenet", include_top=False, input_shape=(224,224,3)
    )
}


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 29s 0us/step
17225924/17225924 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 19s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
history_dict = {}
results = []

for model_name, base_model in models_dict.items():
    print(f"\n🚀 Training {model_name}...\n")

    model = build_tl_model(base_model, num_classes)

    checkpoint = ModelCheckpoint(
        filepath=f"{model_name}_best.h5",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )

    earlystop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=20,
        callbacks=[checkpoint, earlystop]
    )

    history_dict[model_name] = history

    test_loss, test_acc = model.evaluate(test_data)
    results.append([model_name, test_acc])



🚀 Training VGG16...

Epoch 1/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4850 - loss: 1.5366
Epoch 1: val_accuracy improved from None to 0.82509, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 870s 3s/step - accuracy: 0.6437 - loss: 1.0352 - val_accuracy: 0.8251 - val_loss: 1.0018
Epoch 2/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8160 - loss: 0.5474
Epoch 2: val_accuracy improved from 0.82509 to 0.91758, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 522s 2s/step - accuracy: 0.8381 - loss: 0.4879 - val_accuracy: 0.9176 - val_loss: 0.3024
Epoch 3/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8645 - loss: 0.3932
Epoch 3: val_accuracy improved from 0.91758 to 0.94414, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 539s 2s/step - accuracy: 0.8782 - loss: 0.3570 - val_accuracy: 0.9441 - val_loss: 0.1900
Epoch 4/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8938 - loss: 0.3210
Epoch 4: val_accuracy did not improve from 0.94414
260/260 ━━━━━━━━━━━━━━━━━━━━ 539s 2s/step - accuracy: 0.8993 - loss: 0.3062 - val_accuracy: 0.9423 - val_loss: 0.1669
Epoch 5/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9059 - loss: 0.2856
Epoch 5: val_accuracy improved from 0.94414 to 0.95147, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 533s 2s/step - accuracy: 0.9047 - loss: 0.2812 - val_accuracy: 0.9515 - val_loss: 0.1419
Epoch 6/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9109 - loss: 0.2560
Epoch 6: val_accuracy improved from 0.95147 to 0.95604, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 536s 2s/step - accuracy: 0.9133 - loss: 0.2469 - val_accuracy: 0.9560 - val_loss: 0.1374
Epoch 7/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9203 - loss: 0.2392
Epoch 7: val_accuracy improved from 0.95604 to 0.95879, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 424s 2s/step - accuracy: 0.9161 - loss: 0.2469 - val_accuracy: 0.9588 - val_loss: 0.1256
Epoch 8/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9149 - loss: 0.2406
Epoch 8: val_accuracy did not improve from 0.95879
260/260 ━━━━━━━━━━━━━━━━━━━━ 420s 2s/step - accuracy: 0.9226 - loss: 0.2263 - val_accuracy: 0.9588 - val_loss: 0.1081
Epoch 9/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9196 - loss: 0.2361
Epoch 9: val_accuracy improved from 0.95879 to 0.96703, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 428s 2s/step - accuracy: 0.9229 - loss: 0.2263 - val_accuracy: 0.9670 - val_loss: 0.1012
Epoch 10/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9267 - loss: 0.2073
Epoch 10: val_accuracy improved from 0.96703 to 0.97070, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 549s 2s/step - accuracy: 0.9296 - loss: 0.2036 - val_accuracy: 0.9707 - val_loss: 0.0984
Epoch 11/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9272 - loss: 0.2009
Epoch 11: val_accuracy did not improve from 0.97070
260/260 ━━━━━━━━━━━━━━━━━━━━ 564s 2s/step - accuracy: 0.9301 - loss: 0.2001 - val_accuracy: 0.9670 - val_loss: 0.0896
Epoch 12/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9369 - loss: 0.1756
Epoch 12: val_accuracy improved from 0.97070 to 0.97344, saving model to VGG16_best.h5


260/260 ━━━━━━━━━━━━━━━━━━━━ 573s 2s/step - accuracy: 0.9354 - loss: 0.1857 - val_accuracy: 0.9734 - val_loss: 0.0805
Epoch 13/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9382 - loss: 0.1802
Epoch 13: val_accuracy did not improve from 0.97344
260/260 ━━━━━━━━━━━━━━━━━━━━ 566s 2s/step - accuracy: 0.9361 - loss: 0.1895 - val_accuracy: 0.9652 - val_loss: 0.0848
Epoch 14/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9447 - loss: 0.1716
Epoch 14: val_accuracy did not improve from 0.97344
260/260 ━━━━━━━━━━━━━━━━━━━━ 476s 2s/step - accuracy: 0.9388 - loss: 0.1779 - val_accuracy: 0.9734 - val_loss: 0.1050
Epoch 15/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9376 - loss: 0.1777
Epoch 15: val_accuracy did not improve from 0.97344
260/260 ━━━━━━━━━━━━━━━━━━━━ 548s 2s/step - accuracy: 0.9367 - loss: 0.1787 - val_accuracy: 0.9679 - val_loss: 0.0854
Epoch 16/20
260/260 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.9476 - loss: 0.1633
Epoch 16: val_accuracy did n

In [ ]:
results_df = pd.DataFrame(results, columns=["Model", "Test Accuracy"])
results_df = results_df.sort_values(by="Test Accuracy", ascending=False)
print(results_df)

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = tf.keras.models.load_model(f"{best_model_name}_best.h5")

print("Best model:", best_model_name)

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_data])
y_pred = np.argmax(best_model.predict(test_data), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))
